[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/split-pdf.ipynb)

# Chapter 3: PDF Splitting for Large Documents

When working with large PDF documents in RAG systems, it's often necessary to split them into smaller files for processing. This is useful because:

1. **Memory constraints**: Large PDFs may exceed memory limits during parsing
2. **Parallel processing**: Smaller files can be processed concurrently
3. **Incremental updates**: Only re-process changed sections

This notebook demonstrates how to split a PDF into smaller files based on page count using PyPDF2.

**What you'll learn:**
- Download and open PDFs from URLs for processing
- Split large PDFs into smaller files by page range
- Write each page range as a separate PDF file

**Prerequisites:** `pip install PyPDF2 requests`

In [1]:
%pip install --quiet PyPDF2 requests

Note: you may need to restart the kernel to use updated packages.


c:\genai\prodapplication\hands-on-rag\.venv\Scripts\python.exe: No module named pip


In [2]:
import io
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path, PureWindowsPath
from typing import Final
from urllib.parse import urlparse

import requests
from PyPDF2 import PdfReader, PdfWriter
from PyPDF2.errors import PdfReadError

## PDF Splitting Functions

The `split_pdf` function downloads a PDF from a URL and splits it into smaller files:
- `download_pdf` downloads the PDF into memory, checks that it is a PDF, and opens it with `PdfReader`
- `calculate_output_file_count` calculates the number of output files based on `max_pages_per_file`
- `write_page_range_to_pdf` writes each range of pages to a separate PDF file

Progress messages are printed to the cell output with `print`.

In [3]:
'''OPEN A FILE- Open a file- behind the scenes, Python has to locate your file, make sure it exists, and 
ask the operating system for access to the file—after all, you might not have 
permission to access it. we add a path to our filename, which tells 
open where to look for the file. There are actually two kinds 
of paths: a relative path or an absolute one. A relative path is a 
description of the file’s location that is relative to the folder 
(we usually say directory instead when we talk about paths) 
you ran your Python code from.  You can use the .. (two 
periods) notation to specify a directory one level higher.'''
''' READ A FILE- the 
contents of the file are taken and placed into a Python string, which you can 
examine or process as you wish using standard Python string operations.
There are actually two ways to read the contents of your file into 
a string: you can read the entire file all at once, or you can read 
the file a line at a time.
my_file = open('lib.txt', 'r')
my_text = my_file.read()
print(my_text)
my_file.close()
Each line ends with 
an actual new line 
character, otherwise 
known as a newline, 
which is represented by 
the escape character 
sequence ‘\n’.

my_file = open('lib.txt', 'r')
line1 = my_file.readline()
print(line1) 
line2 = my_file.readline()
print(line2)

OR
my_file = open('lib.txt', 'r')
for line in my_file:
    print(line)
my_file.close()
'''




REQUEST_TIMEOUT_SECONDS: Final = 30
DEFAULT_BASE_FILENAME: Final = "output"
PDF_HEADER_MARKER: Final = b"%PDF-"
# PDF readers accept a header that starts anywhere in the first 1024 bytes.
PDF_HEADER_SEARCH_LENGTH_BYTES: Final = 1024

In [4]:
class PdfSplitError(Exception):
    """Base class for failures while downloading or splitting a PDF."""


class InvalidPdfError(PdfSplitError):
    """Raised when the downloaded file is not a PDF or cannot be parsed."""


class PdfOutputError(PdfSplitError):
    """Raised when the output directory or an output file cannot be written."""


@dataclass(frozen=True, kw_only=True, slots=True)
class DownloadedPdf:
    """A PDF opened in memory, with the base name used for its output files."""

    reader: PdfReader
    base_filename: str
    total_pages: int

In [5]:
def get_base_filename_from_url(pdf_url: str) -> str:
    """
    Returns the file name without its extension from the URL path.

    Returns DEFAULT_BASE_FILENAME when the URL path does not end in a file name
    with an extension, for example "https://example.com/download/".
    """
    parsed_url = urlparse(pdf_url)
    url_file_name = PureWindowsPath(parsed_url.path).name

    if url_file_name == "":
        return DEFAULT_BASE_FILENAME

    # Basic check for a file name extension, such as "book.pdf".
    if "." not in url_file_name:
        return DEFAULT_BASE_FILENAME

    base_filename = PureWindowsPath(url_file_name).stem
    return base_filename


def is_pdf_content(file_bytes: bytes) -> bool:
    """Returns True when the PDF header marker is in the first 1024 bytes."""
    header_bytes = file_bytes[:PDF_HEADER_SEARCH_LENGTH_BYTES]
    contains_pdf_header = PDF_HEADER_MARKER in header_bytes
    return contains_pdf_header


def validate_max_pages_per_file(max_pages_per_file: int) -> None:
    """
    Raises:
        ValueError: max_pages_per_file is less than 1.
    """
    if max_pages_per_file < 1:
        raise ValueError(f"max_pages_per_file must be 1 or more, got {max_pages_per_file}.")


def calculate_output_file_count(*, total_pages: int, max_pages_per_file: int) -> int:
    """
    Returns how many files of at most max_pages_per_file pages hold total_pages pages.

    Raises:
        ValueError: max_pages_per_file is less than 1.
    """
    validate_max_pages_per_file(max_pages_per_file)

    # Integer ceiling division: the last file holds the remaining pages.
    output_file_count = (total_pages + max_pages_per_file - 1) // max_pages_per_file
    return output_file_count

In [6]:
def download_pdf(pdf_url: str) -> DownloadedPdf:
    """
    Downloads a PDF from a URL into memory and opens it with PdfReader.

    Returns:
        The PdfReader, the base filename for the output files, and the page count.

    Raises:
        requests.HTTPError: The server returned a 4xx or 5xx status code.
        requests.RequestException: The download failed or timed out.
        InvalidPdfError: The downloaded file is not a PDF or cannot be parsed.
    """
    response = requests.get(pdf_url, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    
    '''
    For your code, you can make the picture more exact:

        - pdf_bytes is a full water tank.
        - pdf_stream is a tap fitted to that tank.
        - Each read(n) opens the tap and lets out n bytes.
        - The position is a counter on the tap that shows how much has come out so far.
    '''

    # The whole PDF is held in memory. This is acceptable for book-sized files.
    pdf_bytes = response.content

    if not is_pdf_content(pdf_bytes):
        content_type = response.headers.get("Content-Type")
        raise InvalidPdfError(
            f"The file downloaded from {pdf_url!r} is not a PDF "
            f"(Content-Type: {content_type!r})."
        )

    pdf_stream = io.BytesIO(pdf_bytes)

    try:
        reader = PdfReader(pdf_stream)
        total_pages = len(reader.pages)
    except PdfReadError as error:
        raise InvalidPdfError(
            f"Could not parse the PDF downloaded from {pdf_url!r}."
        ) from error

    base_filename = get_base_filename_from_url(pdf_url)

    print(f"Successfully downloaded and opened PDF from URL. Total pages: {total_pages}")

    downloaded_pdf = DownloadedPdf(
        reader=reader,
        base_filename=base_filename,
        total_pages=total_pages,
    )
    return downloaded_pdf


def write_page_range_to_pdf(
    *,
    reader: PdfReader,
    start_page_index: int,
    end_page_index: int,
    output_path: Path,
) -> None:
    """
    Writes the pages from start_page_index (inclusive) to end_page_index (exclusive)
    into a new PDF file at output_path. Overwrites output_path if it exists.

    Raises:
        PdfOutputError: The output file could not be written.
    """
    writer = PdfWriter()

    for page_index in range(start_page_index, end_page_index):
        page = reader.pages[page_index]
        writer.add_page(page)

    try:
        with output_path.open("wb") as output_file:
            writer.write(output_file)
    except OSError as error:
        raise PdfOutputError(
            f"Could not write PDF pages to {str(output_path)!r}."
        ) from error

In [7]:
def split_pdf(*, pdf_url: str, output_directory: Path, max_pages_per_file: int) -> None:
    """
    Downloads a PDF from a URL and splits it into multiple smaller PDF files.

    Args:
        pdf_url: The URL of the PDF.
        output_directory: The directory where the output files are saved.
            It is created if it does not exist.
        max_pages_per_file: The maximum number of pages in each output file.
            Must be 1 or more.

    Raises:
        ValueError: max_pages_per_file is less than 1.
        requests.RequestException: The PDF could not be downloaded.
        InvalidPdfError: The downloaded file is not a valid PDF.
        PdfOutputError: The output directory or an output file could not be written.
    """
    # Check the argument before the download, so a bad value fails immediately.
    validate_max_pages_per_file(max_pages_per_file)

    downloaded_pdf = download_pdf(pdf_url)
    reader = downloaded_pdf.reader
    base_filename = downloaded_pdf.base_filename
    total_pages = downloaded_pdf.total_pages

    try:
        output_directory.mkdir(parents=True, exist_ok=True)
    except OSError as error:
        raise PdfOutputError(
            f"Could not create output directory {str(output_directory)!r}."
        ) from error

    print(f"Output directory '{output_directory}' ensured.")

    output_file_count = calculate_output_file_count(
        total_pages=total_pages,
        max_pages_per_file=max_pages_per_file,
    )
    print(f"Splitting into {output_file_count} files of max {max_pages_per_file} pages each.")

    for file_index in range(output_file_count):
        file_number = file_index + 1
        start_page_index = file_index * max_pages_per_file
        # The last file can have fewer pages than max_pages_per_file.
        end_page_index = min(start_page_index + max_pages_per_file, total_pages)

        first_page_number = start_page_index + 1
        last_page_number = end_page_index
        print(
            f"Processing file {file_number}/{output_file_count} "
            f"(pages {first_page_number}-{last_page_number})..."
        )

        output_file_name = f"{base_filename}_pages_{first_page_number}-{last_page_number}.pdf"
        output_path = output_directory / output_file_name

        write_page_range_to_pdf(
            reader=reader,
            start_page_index=start_page_index,
            end_page_index=end_page_index,
            output_path=output_path,
        )

        print(f"File {file_number} saved as '{output_path}'")

    print("\nPDF splitting completed successfully!")

Let's split the Sutton & Barto Reinforcement Learning textbook (352 pages) into files of at most 50 pages each.

In [9]:
book_pdf_url = (
    "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
)
book_output_directory = Path("pdf_split_files")
book_max_pages_per_file = 50

split_pdf(
    pdf_url=book_pdf_url,
    output_directory=book_output_directory,
    max_pages_per_file=book_max_pages_per_file,
)

Successfully downloaded and opened PDF from URL. Total pages: 352
Output directory 'pdf_split_files' ensured.
Splitting into 8 files of max 50 pages each.
Processing file 1/8 (pages 1-50)...
File 1 saved as 'pdf_split_files\SuttonBartoIPRLBook2ndEd_pages_1-50.pdf'
Processing file 2/8 (pages 51-100)...
File 2 saved as 'pdf_split_files\SuttonBartoIPRLBook2ndEd_pages_51-100.pdf'
Processing file 3/8 (pages 101-150)...
File 3 saved as 'pdf_split_files\SuttonBartoIPRLBook2ndEd_pages_101-150.pdf'
Processing file 4/8 (pages 151-200)...
File 4 saved as 'pdf_split_files\SuttonBartoIPRLBook2ndEd_pages_151-200.pdf'
Processing file 5/8 (pages 201-250)...
File 5 saved as 'pdf_split_files\SuttonBartoIPRLBook2ndEd_pages_201-250.pdf'
Processing file 6/8 (pages 251-300)...
File 6 saved as 'pdf_split_files\SuttonBartoIPRLBook2ndEd_pages_251-300.pdf'
Processing file 7/8 (pages 301-350)...
File 7 saved as 'pdf_split_files\SuttonBartoIPRLBook2ndEd_pages_301-350.pdf'
Processing file 8/8 (pages 351-352)...
Fi